In [1]:
import pandas as pd
import numpy as np
import warnings, json, re, time, glob, os
warnings.filterwarnings('ignore')

from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.neighbors import NearestNeighbors
import folium
from folium.plugins import HeatMap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from collections import defaultdict

# Shapely for convex hull polygons
try:
    from shapely.geometry import MultiPoint
    SHAPELY_AVAILABLE = True
except ImportError:
    print("  [WARN] shapely not installed. Run: pip install shapely")
    print("         Convex hull polygons will be skipped.")
    SHAPELY_AVAILABLE = False


In [2]:
EPSILON_KM   = 5          # default;
MIN_SAMPLES  = 3
EARTH_RADIUS = 6371.0

# Financial assumptions
BOOKING_VALUE_SCENARIOS = {
    'Conservative (₹8K)':  8_000,
    'Base Case (₹15K)':   15_000,
    'Optimistic (₹25K)':  25_000,
}
AVG_BOOKING_VALUE      = 15_000
REFUND_PROCESSING_COST = 200
PER_REFUND_LOSS        = AVG_BOOKING_VALUE + REFUND_PROCESSING_COST

SEVERITY_THRESHOLDS = {'Critical': 20, 'High': 10, 'Medium': 5}

SEVERITY_COLORS = {
    'Critical': '#dc2626',
    'High':     '#ea580c',
    'Medium':   '#f59e0b',
    'Low':      '#10b981',
}


In [3]:

# 1. DATA LOADING
print("=" * 70)
print("COOX NON-SERVICEABLE BOOKINGS ANALYSIS v3")
print("=" * 70)

import os
FILE_PATH = r"/content/IIT Roorkee __ COOX.xlsx"

if not FILE_PATH:
    raise ValueError("FILE_PATH is empty. Paste your dataset path at the top of the script.")

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"File not found: '{FILE_PATH}'\nCheck the path and try again.")

ext = os.path.splitext(FILE_PATH)[1].lower()
if ext == '.xlsx':
    df = pd.read_excel(FILE_PATH)
elif ext == '.csv':
    df = pd.read_csv(FILE_PATH)
else:
    raise ValueError(f"Unsupported file type '{ext}'. Must be .xlsx or .csv.")

print(f" Loaded: {FILE_PATH}  ({len(df)} rows)")

print(f"\n[1] DATA OVERVIEW")
print(f"    Total records : {len(df)}")
print(f"    Columns       : {df.columns.tolist()}")

# Normalise column names
df.columns = df.columns.str.strip()

df_clean = df.dropna(subset=['Lat', 'Long']).copy()
df_clean['Area']         = df_clean['Area'].fillna('Unknown')
df_clean['Address Type'] = df_clean['Address Type'].fillna('Unknown')
df_clean['City']         = df_clean['City'].str.strip().str.title()
df_clean['State']        = df_clean['State'].str.strip().str.title()

print(f"    After cleaning : {len(df_clean)} records  |  Dropped: {len(df) - len(df_clean)}")
print(f"\n    City distribution:\n{df_clean['City'].value_counts().to_string()}")


COOX NON-SERVICEABLE BOOKINGS ANALYSIS v3
 Loaded: /content/IIT Roorkee __ COOX.xlsx  (557 rows)

[1] DATA OVERVIEW
    Total records : 557
    Columns       : ['Booking ID', 'Payment Status', 'Address ID', 'Address Type', 'Area', 'City', 'State', 'Country', 'Lat', 'Long']
    After cleaning : 546 records  |  Dropped: 11

    City distribution:
City
Bengaluru                     143
Pune                           94
Hyderabad                      68
Chennai                        66
Ahmedabad                      38
Jaipur                         25
Kolkata                        18
Surat                          15
Navi Mumbai                    14
Indore                         11
Chandigarh                      9
Mumbai                          9
Lucknow                         8
Thane                           6
Gurugram                        6
Sahibzada Ajit Singh Nagar      4
Panaji                          4
Vadodara                        3
Delhi                           2
Ul

In [4]:
#Auto-detect date column for temporal analysis
DATE_COL = None
for candidate in ['Booking Date', 'Date', 'Created At', 'Order Date', 'booking_date']:
    if candidate in df_clean.columns:
        DATE_COL = candidate
        df_clean[DATE_COL] = pd.to_datetime(df_clean[DATE_COL], errors='coerce')
        df_clean['YearMonth'] = df_clean[DATE_COL].dt.to_period('M')
        print(f"    Temporal column : '{DATE_COL}' detected — trend analysis enabled")
        break
if DATE_COL is None:
    print("    Temporal column : NOT FOUND — trend analysis will be skipped")

    Temporal column : NOT FOUND — trend analysis will be skipped


In [5]:
# 2. PINCODE EXTRACTION
print(f"\n[2] PINCODE EXTRACTION")
_PINCODE_RE = re.compile(r'\b[1-9][0-9]{5}\b')

def pincode_from_text(text: str):
    m = _PINCODE_RE.findall(str(text))
    return m[0] if m else None

df_clean['Pincode'] = df_clean['Area'].apply(pincode_from_text)

regex_found = df_clean['Pincode'].notna().sum()
print(f"    Regex extraction : {regex_found} / {len(df_clean)}")

missing_mask   = df_clean['Pincode'].isna()
unique_missing = df_clean.loc[missing_mask, ['Lat', 'Long']].drop_duplicates()
print(f"    Reverse geocoding {len(unique_missing)} unique coords…")
SKIP_GEOCODING = False

if SKIP_GEOCODING:
    print("      SKIPPING geocoding (SKIP_GEOCODING=True)")
    coord_pin_cache = {}
else:
    geolocator = Nominatim(user_agent="coox_analysis_v3", timeout=5)
    coord_pin_cache = {}
    total_coords = len(unique_missing)
    errors = 0
    for i, (_, row) in enumerate(unique_missing.iterrows(), 1):
        if i % 50 == 0:
            print(f"      Geocoded {i}/{total_coords}  ({errors} errors so far)…")
        key = (round(row['Lat'], 5), round(row['Long'], 5))
        try:
            loc = geolocator.reverse(f"{key[0]},{key[1]}", language='en')
            if loc and loc.raw.get('address'):
                coord_pin_cache[key] = loc.raw['address'].get('postcode')

            time.sleep(0.3)
        except Exception:
            coord_pin_cache[key] = None
            errors += 1
    print(f"      Done. {total_coords} coords processed, {errors} errors.")

def lookup_cache(row):
    if pd.notna(row['Pincode']):
        return row['Pincode']
    key = (round(row['Lat'], 5), round(row['Long'], 5))
    return coord_pin_cache.get(key)

df_clean['Pincode'] = df_clean.apply(lookup_cache, axis=1)
df_clean['Pincode'] = df_clean['Pincode'].fillna('N/A')

total_found = (df_clean['Pincode'] != 'N/A').sum()
print(f"    After geocoding  : {total_found} / {len(df_clean)} pincodes resolved")



[2] PINCODE EXTRACTION
    Regex extraction : 94 / 546
    Reverse geocoding 369 unique coords…
      Geocoded 50/369  (29 errors so far)…
      Geocoded 100/369  (79 errors so far)…
      Geocoded 150/369  (129 errors so far)…
      Geocoded 200/369  (179 errors so far)…
      Geocoded 250/369  (212 errors so far)…
      Geocoded 300/369  (262 errors so far)…
      Geocoded 350/369  (312 errors so far)…
      Done. 369 coords processed, 332 errors.
    After geocoding  : 130 / 546 pincodes resolved


In [6]:
# 3. EPSILON CROSS-VALIDATION
print(f"\n[3] EPSILON SELECTION & CROSS-VALIDATION")

coords_rad = np.radians(df_clean[['Lat', 'Long']].values)

# 3a.K-distance plot (k = MIN_SAMPLES - 1 = 2nd nearest neighbor)
k = MIN_SAMPLES - 1
nbrs = NearestNeighbors(n_neighbors=k, metric='haversine', algorithm='ball_tree')
nbrs.fit(coords_rad)
distances, _ = nbrs.kneighbors(coords_rad)
kth_distances_km = np.sort(distances[:, -1]) * EARTH_RADIUS   # convert to km


fig_kdist = go.Figure()
fig_kdist.add_trace(go.Scatter(
    y=kth_distances_km,
    mode='lines',
    line=dict(color='#7c3aed', width=2),
    name=f'Distance to {k}th nearest neighbor'
))
# Mark the "elbow" suggestion at the inflection point
diffs = np.diff(kth_distances_km)
elbow_idx = int(np.argmax(diffs > np.percentile(diffs, 90)))
elbow_val = round(kth_distances_km[elbow_idx], 2)
fig_kdist.add_hline(
    y=elbow_val, line_dash='dash', line_color='red',
    annotation_text=f'Suggested ε ≈ {elbow_val} km',
    annotation_position='top left'
)
fig_kdist.add_hline(
    y=EPSILON_KM, line_dash='dot', line_color='orange',
    annotation_text=f'Current ε = {EPSILON_KM} km',
    annotation_position='bottom right'
)
fig_kdist.update_layout(
    title=f'K-Distance Plot (k={k}) — Elbow Suggests ε ≈ {elbow_val} km',
    xaxis_title='Points (sorted by distance)',
    yaxis_title=f'Distance to {k}th Neighbor (km)',
    paper_bgcolor='#0f0f0f', plot_bgcolor='#1a1a1a',
    font=dict(color='#e5e5e5'),
    height=450
)
fig_kdist.write_html('coox_kdistance_plot.html')
print(f"    K-distance elbow suggests ε ≈ {elbow_val} km  (current setting: {EPSILON_KM} km)")
print(f"     coox_kdistance_plot.html")



[3] EPSILON SELECTION & CROSS-VALIDATION
    K-distance elbow suggests ε ≈ 0.46 km  (current setting: 5 km)
     coox_kdistance_plot.html


In [7]:
#3b.Multi-epsilon cross-validation
epsilon_candidates = [3, 5, 8]
cv_results = []

print(f"\n    Cross-validating ε ∈ {epsilon_candidates} km:")
for eps_km in epsilon_candidates:
    eps_rad = eps_km / EARTH_RADIUS
    db = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES,
                algorithm='ball_tree', metric='haversine')
    labels = db.fit_predict(coords_rad)
    n_cl   = int(labels.max()) + 1
    n_ns   = int((labels == -1).sum())
    coverage_pct = round((len(labels) - n_ns) / len(labels) * 100, 1)

    sil = None
    cl_mask = labels >= 0
    if n_cl > 1 and cl_mask.sum() < 5000:
        dist_mat = pairwise_distances(coords_rad[cl_mask], metric='haversine') * EARTH_RADIUS
        sil = round(silhouette_score(dist_mat, labels[cl_mask], metric='precomputed'), 3)

    cv_results.append({
        'Epsilon_km':  eps_km,
        'N_clusters':  n_cl,
        'N_noise':     n_ns,
        'Coverage_pct': coverage_pct,
        'Silhouette':  sil,
    })
    print(f"      ε={eps_km} km → {n_cl} clusters, {n_ns} noise, "
          f"coverage={coverage_pct}%, silhouette={sil}")

cv_df = pd.DataFrame(cv_results)

# Pick epsilon with best silhouette
valid_cv = cv_df.dropna(subset=['Silhouette'])
if not valid_cv.empty:
    best_eps = valid_cv.loc[valid_cv['Silhouette'].idxmax(), 'Epsilon_km']
    best_sil = valid_cv['Silhouette'].max()
    print(f"\n    Best ε = {best_eps} km  (silhouette = {best_sil})")
    EPSILON_KM = int(best_eps)
else:
    print(f"\n    Could not compute silhouette — keeping ε = {EPSILON_KM} km")



    Cross-validating ε ∈ [3, 5, 8] km:
      ε=3 km → 63 clusters, 235 noise, coverage=57.0%, silhouette=0.755
      ε=5 km → 43 clusters, 139 noise, coverage=74.5%, silhouette=0.515
      ε=8 km → 32 clusters, 86 noise, coverage=84.2%, silhouette=0.484

    Best ε = 3 km  (silhouette = 0.755)


In [8]:
# 4. FINAL DBSCAN  (with best epsilon)
print(f"\n[4] DBSCAN CLUSTERING  (ε={EPSILON_KM} km, min_samples={MIN_SAMPLES})")

eps_rad = EPSILON_KM / EARTH_RADIUS
dbscan  = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES,
                 algorithm='ball_tree', metric='haversine')
df_clean['Cluster'] = dbscan.fit_predict(coords_rad)

n_clusters  = int(df_clean['Cluster'].max()) + 1
n_noise     = int((df_clean['Cluster'] == -1).sum())
n_clustered = len(df_clean) - n_noise

print(f"    Clusters found   : {n_clusters}")
print(f"    Clustered pts    : {n_clustered}  ({n_clustered/len(df_clean)*100:.1f}%)")
print(f"    Noise (isolated) : {n_noise}  ({n_noise/len(df_clean)*100:.1f}%)")

clustered_df = df_clean[df_clean['Cluster'] >= 0]
if n_clusters > 1 and len(clustered_df) < 5000:
    coords_cl = np.radians(clustered_df[['Lat', 'Long']].values)
    dist_mat  = pairwise_distances(coords_cl, metric='haversine') * EARTH_RADIUS
    sil = silhouette_score(dist_mat, clustered_df['Cluster'], metric='precomputed')
    print(f"    Silhouette score : {sil:.3f}  (>0.5 = well-separated)")



[4] DBSCAN CLUSTERING  (ε=3 km, min_samples=3)
    Clusters found   : 63
    Clustered pts    : 311  (57.0%)
    Noise (isolated) : 235  (43.0%)
    Silhouette score : 0.755  (>0.5 = well-separated)


In [9]:
# 5. CLUSTER SUMMARY + CONVEX HULL POLYGONS
print(f"\n[5] BUILDING CLUSTER SUMMARY  (with convex hull polygons)")

def classify_severity(n: int) -> str:
    if n >= SEVERITY_THRESHOLDS['Critical']: return 'Critical'
    if n >= SEVERITY_THRESHOLDS['High']:     return 'High'
    if n >= SEVERITY_THRESHOLDS['Medium']:   return 'Medium'
    return 'Low'

rows = []
hull_polygons = {}   # cluster_id → list of [lat, lng] vertices

for cid in range(n_clusters):
    grp  = df_clean[df_clean['Cluster'] == cid]
    cnt  = len(grp)
    pts  = grp[['Lat', 'Long']].values

    clat, clng = pts[:, 0].mean(), pts[:, 1].mean()

    # 95th-percentile radius
    dists  = grp.apply(lambda r: geodesic((clat, clng), (r['Lat'], r['Long'])).km, axis=1)
    radius = round(np.percentile(dists, 95), 2)

    # Convex hull polygon
    hull_coords = None
    if SHAPELY_AVAILABLE and len(pts) >= 3:
        try:
            mp    = MultiPoint(pts)          # (lat, lng) points
            hull  = mp.convex_hull
            if hull.geom_type == 'Polygon':
                # exterior coords as [[lat, lng], …]
                hull_coords = [[round(y, 6), round(x, 6)]
                               for x, y in hull.exterior.coords]

                hull_coords = [[round(lat, 6), round(lng, 6)]
                               for lat, lng in hull.exterior.coords]
        except Exception:
            hull_coords = None
    hull_polygons[cid] = hull_coords

    pincodes  = sorted(grp.loc[grp['Pincode'] != 'N/A', 'Pincode'].unique().tolist())
    top_city  = grp['City'].mode()[0]
    top_state = grp['State'].mode()[0] if 'State' in grp.columns else ''
    addr_types = grp['Address Type'].value_counts().index.tolist()[:3]
    severity  = classify_severity(cnt)

    rows.append({
        'Cluster_ID':       cid,
        'Booking_Count':    cnt,
        'Severity':         severity,
        'Top_City':         top_city,
        'Top_State':        top_state,
        'Centroid_Lat':     round(clat, 6),
        'Centroid_Long':    round(clng, 6),
        'Radius_km':        radius,
        'Hull_Available':   hull_coords is not None,
        'All_Pincodes':     ', '.join(pincodes) if pincodes else 'N/A',
        'Num_Pincodes':     len(pincodes),
        'Primary_Pincode':  pincodes[0] if pincodes else 'N/A',
        'Address_Types':    ', '.join(addr_types),
        'Sample_Area':      str(grp['Area'].iloc[0])[:80],
        'Estimated_Loss_INR': cnt * PER_REFUND_LOSS,
    })

cluster_df = pd.DataFrame(rows).sort_values('Booking_Count', ascending=False).reset_index(drop=True)

for sev in ['Critical', 'High', 'Medium', 'Low']:
    sub = cluster_df[cluster_df['Severity'] == sev]
    print(f"    {sev:8s}: {len(sub):2d} clusters  |  {sub['Booking_Count'].sum():3d} bookings  |  ₹{sub['Estimated_Loss_INR'].sum():,}")



[5] BUILDING CLUSTER SUMMARY  (with convex hull polygons)
    Critical:  0 clusters  |    0 bookings  |  ₹0
    High    :  6 clusters  |   77 bookings  |  ₹1,170,400
    Medium  : 19 clusters  |  112 bookings  |  ₹1,702,400
    Low     : 38 clusters  |  122 bookings  |  ₹1,854,400


In [10]:
# 6. NOISE / ISOLATED BOOKINGS
print(f"\n[6] ISOLATED BOOKINGS ANALYSIS")
noise_df = df_clean[df_clean['Cluster'] == -1].copy()

noise_pin_counts = (
    noise_df[noise_df['Pincode'] != 'N/A']
    .groupby('Pincode')
    .agg(Count=('Booking ID', 'count'),
         Cities=('City', lambda x: ', '.join(x.unique())))
    .query('Count >= 2')
    .sort_values('Count', ascending=False)
    .reset_index()
)
noise_pin_counts['Recommendation'] = 'Monitor — block if count reaches 3'

print(f"    Isolated bookings : {len(noise_df)}")
print(f"    Watchlist pins    : {len(noise_pin_counts)} pincodes with 2+ isolated hits")



[6] ISOLATED BOOKINGS ANALYSIS
    Isolated bookings : 235
    Watchlist pins    : 11 pincodes with 2+ isolated hits


In [11]:
# 7. FINANCIAL SENSITIVITY ANALYSIS
print(f"\n[7] FINANCIAL SENSITIVITY ANALYSIS")
print(f"    {'Scenario':<25} {'Total Loss':>14}  {'Cluster Loss':>14}  {'Recoverable %':>13}")
print(f"    {'-'*70}")

sensitivity_rows = []
for scenario_name, bv in BOOKING_VALUE_SCENARIOS.items():
    loss_per = bv + REFUND_PROCESSING_COST
    tot      = len(df_clean) * loss_per
    clu      = cluster_df['Booking_Count'].sum() * loss_per
    pct      = round(clu / tot * 100, 1)
    print(f"    {scenario_name:<25} ₹{tot:>12,}  ₹{clu:>12,}  {pct:>12}%")
    sensitivity_rows.append({'Scenario': scenario_name, 'Total_Loss': tot,
                             'Cluster_Loss': clu, 'Recoverable_Pct': pct})

sensitivity_df = pd.DataFrame(sensitivity_rows)

total_loss   = len(df_clean) * PER_REFUND_LOSS
cluster_loss = cluster_df['Estimated_Loss_INR'].sum()



[7] FINANCIAL SENSITIVITY ANALYSIS
    Scenario                      Total Loss    Cluster Loss  Recoverable %
    ----------------------------------------------------------------------
    Conservative (₹8K)        ₹   4,477,200  ₹   2,550,200          57.0%
    Base Case (₹15K)          ₹   8,299,200  ₹   4,727,200          57.0%
    Optimistic (₹25K)         ₹  13,759,200  ₹   7,837,200          57.0%


In [12]:
# 8. BUSINESS IMPACT BY PHASE
print(f"\n[8] BUSINESS IMPACT BY PHASE")

phase_labels = ['Critical', 'High', 'Medium', 'Low']
phase_weeks  = ['Week 1', 'Weeks 2–3', 'Month 1', 'Month 2+']
cumulative   = 0
phase_summary = []

for sev, week in zip(phase_labels, phase_weeks):
    sub  = cluster_df[cluster_df['Severity'] == sev]
    loss = int(sub['Estimated_Loss_INR'].sum())
    cumulative += loss
    pincodes_flat = [
        p.strip() for ps in sub['All_Pincodes']
        for p in ps.split(',') if p.strip() not in ('N/A', '')
    ]
    phase_summary.append({
        'Phase':        f'{sev} ({week})',
        'Clusters':     len(sub),
        'Bookings':     int(sub['Booking_Count'].sum()),
        'Loss_INR':     loss,
        'Cumulative':   cumulative,
        'Pct_of_Total': round(loss / total_loss * 100, 1),
        'Pincodes':     sorted(set(pincodes_flat)),
    })
    print(f"    {sev:8s} ({week:12s}): ₹{loss:>10,}  |  {len(set(pincodes_flat))} unique pincodes")

phase_df = pd.DataFrame(phase_summary)
print(f"\n    Total addressable loss : ₹{total_loss:,}")
print(f"    Cluster-covered loss   : ₹{cluster_loss:,}  ({cluster_loss/total_loss*100:.1f}%)")



[8] BUSINESS IMPACT BY PHASE
    Critical (Week 1      ): ₹         0  |  0 unique pincodes
    High     (Weeks 2–3   ): ₹ 1,170,400  |  7 unique pincodes
    Medium   (Month 1     ): ₹ 1,702,400  |  14 unique pincodes
    Low      (Month 2+    ): ₹ 1,854,400  |  14 unique pincodes

    Total addressable loss : ₹8,299,200
    Cluster-covered loss   : ₹4,727,200  (57.0%)


In [13]:
# 9. TEMPORAL TREND ANALYSIS
if DATE_COL:
    print(f"\n[9] TEMPORAL TREND ANALYSIS")

    monthly = (
        df_clean.groupby('YearMonth')
        .agg(Total_Bookings=('Booking ID', 'count'))
        .reset_index()
    )
    clustered_monthly = (
        df_clean[df_clean['Cluster'] >= 0]
        .groupby('YearMonth')
        .agg(Clustered_Outskirts=('Booking ID', 'count'))
        .reset_index()
    )
    trend_df = monthly.merge(clustered_monthly, on='YearMonth', how='left').fillna(0)
    trend_df['Outskirt_Rate_Pct'] = (
        trend_df['Clustered_Outskirts'] / trend_df['Total_Bookings'] * 100
    ).round(1)

    print(trend_df.to_string(index=False))

    fig_trend = make_subplots(specs=[[{'secondary_y': True}]])
    months_str = trend_df['YearMonth'].astype(str).tolist()

    fig_trend.add_trace(go.Bar(
        x=months_str, y=trend_df['Total_Bookings'].tolist(),
        name='Total Bookings', marker_color='#7c3aed', opacity=0.7,
    ), secondary_y=False)

    fig_trend.add_trace(go.Scatter(
        x=months_str, y=trend_df['Clustered_Outskirts'].tolist(),
        name='Outskirt Bookings', mode='lines+markers',
        line=dict(color='#dc2626', width=2),
    ), secondary_y=False)

    fig_trend.add_trace(go.Scatter(
        x=months_str, y=trend_df['Outskirt_Rate_Pct'].tolist(),
        name='Outskirt Rate (%)', mode='lines+markers',
        line=dict(color='#f59e0b', width=2, dash='dot'),
    ), secondary_y=True)

    fig_trend.update_layout(
        title='Monthly Outskirt Booking Trend',
        paper_bgcolor='#0f0f0f', plot_bgcolor='#1a1a1a',
        font=dict(color='#e5e5e5'), height=450, barmode='overlay',
    )
    fig_trend.update_yaxes(title_text='Booking Count',  secondary_y=False, gridcolor='#333')
    fig_trend.update_yaxes(title_text='Outskirt Rate %', secondary_y=True,  gridcolor='#333')
    fig_trend.write_html('coox_temporal_trend.html')
    print(f"     coox_temporal_trend.html")
    trend_df.to_csv('coox_monthly_trend.csv', index=False)
else:
    print(f"\n[9] TEMPORAL TREND ANALYSIS — skipped (no date column found)")


[9] TEMPORAL TREND ANALYSIS — skipped (no date column found)


In [14]:
# 10. ADDRESS TYPE ANALYSIS
print(f"\n[10] ADDRESS TYPE ANALYSIS")
addr_stats = (
    df_clean.groupby('Address Type')
    .agg(Total=('Booking ID', 'count'),
         In_Clusters=('Cluster', lambda x: (x >= 0).sum()))
    .assign(Cluster_Rate=lambda d: (d['In_Clusters'] / d['Total'] * 100).round(1))
    .sort_values('Total', ascending=False)
)
print(addr_stats.to_string())


[10] ADDRESS TYPE ANALYSIS
              Total  In_Clusters  Cluster_Rate
Address Type                                  
Home            441          264          59.9
Farmhouse        75           26          34.7
Other            11            8          72.7
Resort           10            7          70.0
Office            9            6          66.7


In [15]:
# 11. SAVE OUTPUTS
print(f"\n[11] SAVING OUTPUT FILES")

cluster_df.to_csv('coox_cluster_blocking_list.csv', index=False)
df_clean.to_csv('coox_bookings_with_clusters.csv', index=False)
noise_pin_counts.to_csv('coox_isolated_pincode_watchlist.csv', index=False)
cv_df.to_csv('coox_epsilon_crossvalidation.csv', index=False)
sensitivity_df.to_csv('coox_financial_sensitivity.csv', index=False)

all_block_pins = sorted(set(
    p.strip()
    for ps in cluster_df['All_Pincodes']
    for p in ps.split(',')
    if p.strip() not in ('N/A', '')
))
phase1_pins = phase_summary[0]['Pincodes']
phase2_pins = phase_summary[1]['Pincodes']

# Hull polygon export
hull_export = []
for cid, coords in hull_polygons.items():
    row_meta = cluster_df[cluster_df['Cluster_ID'] == cid]
    hull_export.append({
        'cluster_id':    cid,
        'severity':      row_meta['Severity'].values[0] if len(row_meta) else 'Unknown',
        'top_city':      row_meta['Top_City'].values[0] if len(row_meta) else 'Unknown',
        'booking_count': int(row_meta['Booking_Count'].values[0]) if len(row_meta) else 0,
        'pincodes':      row_meta['All_Pincodes'].values[0].split(', ') if len(row_meta) else [],
        'centroid':      {'lat': float(row_meta['Centroid_Lat'].values[0]),
                          'lng': float(row_meta['Centroid_Long'].values[0])} if len(row_meta) else {},
        'radius_km':     float(row_meta['Radius_km'].values[0]) if len(row_meta) else 0.0,
        'hull_polygon':  coords,
        'hull_note':     'Use hull_polygon for geoblocking; radius_km is approximate fallback.'
    })

# Convert NumPy types in phase_summary for JSON serialization
serializable_phase_summary = []
for phase in phase_summary:
    new_phase = phase.copy()
    if isinstance(new_phase.get('Pct_of_Total'), (np.float64, np.float32)):
        new_phase['Pct_of_Total'] = float(new_phase['Pct_of_Total'])
    serializable_phase_summary.append(new_phase)

# Convert NumPy types in sensitivity_rows for JSON serialization
serializable_sensitivity_rows = []
for row in sensitivity_rows:
    new_row = row.copy()
    if isinstance(new_row.get('Cluster_Loss'), (np.int64, np.int32)):
        new_row['Cluster_Loss'] = int(new_row['Cluster_Loss'])
    if isinstance(new_row.get('Recoverable_Pct'), (np.float64, np.float32)):
        new_row['Recoverable_Pct'] = float(new_row['Recoverable_Pct'])
    serializable_sensitivity_rows.append(new_row)

# Convert NumPy types in cv_results for JSON serialization
serializable_cv_results = []
for cv_res in cv_results:
    new_cv_res = cv_res.copy()
    if new_cv_res['Silhouette'] is not None and isinstance(new_cv_res['Silhouette'], (np.float64, np.float32)):
        new_cv_res['Silhouette'] = float(new_cv_res['Silhouette'])
    serializable_cv_results.append(new_cv_res)

summary_json = {
    'generated_at':         pd.Timestamp.now().isoformat(),
    'epsilon_km_used':      EPSILON_KM,
    'epsilon_note':         f'Auto-selected via silhouette cross-validation. K-distance elbow suggested {elbow_val} km.',
    'financial_assumption': f'₹{AVG_BOOKING_VALUE:,} avg booking value + ₹{REFUND_PROCESSING_COST} processing',
    'total_bookings':       len(df_clean),
    'total_clusters':       n_clusters,
    'clustered_bookings':   n_clustered,
    'noise_bookings':       n_noise,
    'unique_pincodes_all':  all_block_pins,
    'unique_pincodes_p1':   phase1_pins,
    'unique_pincodes_p2':   phase2_pins,
    'watchlist_pincodes':   noise_pin_counts['Pincode'].tolist(),
    'total_loss_inr':       total_loss,
    'cluster_loss_inr':     int(cluster_loss),
    'phase_summary':        serializable_phase_summary,
    'sensitivity_analysis': serializable_sensitivity_rows,
    'epsilon_cv':           serializable_cv_results,
    'clusters':             hull_export,   # includes hull_polygon per cluster
}
with open('coox_analysis_summary.json', 'w') as f:
    json.dump(summary_json, f, indent=2)

print(f"    coox_cluster_blocking_list.csv   ({len(cluster_df)} clusters)")
print(f"    coox_bookings_with_clusters.csv  ({len(df_clean)} rows)")
print(f"    coox_isolated_pincode_watchlist.csv ({len(noise_pin_counts)} pins)")
print(f"    coox_epsilon_crossvalidation.csv")
print(f"    coox_financial_sensitivity.csv")
print(f"    coox_analysis_summary.json  (includes hull_polygon per cluster)")

# 12. VISUALISATIONS
print(f"\n[12] GENERATING VISUALISATIONS")

SEV_ICON_COLOR = {'Critical': 'red', 'High': 'orange', 'Medium': 'beige', 'Low': 'green'}

#12a.All-India cluster map
india_map = folium.Map(location=[20.5, 78.9], zoom_start=5,
                       tiles='CartoDB positron')

# Heatmap
heat_data = [[row['Lat'], row['Long'], 1] for _, row in df_clean.iterrows()]
HeatMap(
    heat_data,
    radius=18,
    blur=12,
    min_opacity=0.4,
    max_zoom=14,
    name='Heatmap',
).add_to(india_map)

cluster_layer = folium.FeatureGroup(name='Cluster circles (approx)').add_to(india_map)
hull_layer    = folium.FeatureGroup(name='Convex hull polygons (precise)').add_to(india_map)

for _, row in cluster_df.iterrows():
    cid    = row['Cluster_ID']
    color  = SEVERITY_COLORS[row['Severity']]
    radius = max(3000, row['Radius_km'] * 1000)
    popup  = (
        f"<b>Cluster {cid}</b><br>"
        f"<b>{row['Severity']}</b> — {row['Booking_Count']} bookings<br>"
        f"City: {row['Top_City']}<br>"
        f"Pincodes: {row['All_Pincodes']}<br>"
        f"Est. loss: ₹{row['Estimated_Loss_INR']:,}<br>"
        f"Radius: ~{row['Radius_km']} km<br>"
        f"Hull: {'Available' if row['Hull_Available'] else ' N/A (< 3 pts)'}"
    )
    folium.Circle(
        location=[row['Centroid_Lat'], row['Centroid_Long']],
        radius=radius, color=color, fill=True, fill_opacity=0.18, weight=2,
        popup=folium.Popup(popup, max_width=300),
    ).add_to(cluster_layer)

    # Draw convex hull polygon
    hull_coords = hull_polygons.get(cid)
    if hull_coords and len(hull_coords) >= 3:
        folium.Polygon(
            locations=hull_coords,
            color=color, fill=True, fill_opacity=0.30, weight=2,
            popup=folium.Popup(f"<b>Hull — Cluster {cid}</b><br>{row['Booking_Count']} bookings<br>"
                               f"Severity: {row['Severity']}<br>Pincodes: {row['All_Pincodes']}",
                               max_width=280),
            tooltip=f"Cluster {cid} — {row['Severity']}",
        ).add_to(hull_layer)

noise_layer = folium.FeatureGroup(name='Isolated bookings', show=False).add_to(india_map)
for _, row in noise_df.iterrows():
    folium.CircleMarker(
        [row['Lat'], row['Long']], radius=3,
        color='#94a3b8', fill=True, fill_opacity=0.5, weight=0,
        popup=f"{row['Area']} ({row['City']})",
    ).add_to(noise_layer)

legend_html = """
<div style="position:fixed;top:12px;right:12px;z-index:9999;
     background:white;padding:14px 18px;border-radius:8px;
     border:1px solid #ccc;font-family:Arial;font-size:13px;
     box-shadow:2px 2px 6px rgba(0,0,0,.2)">
  <b>COOX Blocking Map v3</b><br><br>
  <span style="color:#dc2626">●</span> Critical (≥20)<br>
  <span style="color:#ea580c">●</span> High (10–19)<br>
  <span style="color:#f59e0b">●</span> Medium (5–9)<br>
  <span style="color:#10b981">●</span> Low (3–4)<br>
  <span style="color:#94a3b8">●</span> Isolated<br><br>
  <i>Filled polygons = convex hulls<br>
  Circles = radius approximation</i>
</div>"""
india_map.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(india_map)
india_map.save('coox_cluster_map.html')
print("     coox_cluster_map.html  (includes convex hull polygons)")

#12b. City-level maps
cities_with_clusters = cluster_df['Top_City'].str.strip().str.title().unique().tolist()
df_clean['City'] = df_clean['City'].str.strip().str.title()

for city in cities_with_clusters:
    city_data  = df_clean[df_clean['City'] == city]
    city_clust = cluster_df[cluster_df['Top_City'].str.strip().str.title() == city]

    if city_data.empty:
        print(f"      No rows found for city '{city}' — skipping")
        continue

    center = [city_data['Lat'].mean(), city_data['Long'].mean()]
    m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')

    # Heatmap layer
    city_heat = [[row['Lat'], row['Long'], 1] for _, row in city_data.iterrows()]
    HeatMap(city_heat, radius=22, blur=12, min_opacity=0.4).add_to(m)


    for _, row in city_clust.iterrows():
        cid   = row['Cluster_ID']
        color = SEVERITY_COLORS[row['Severity']]
        folium.Circle(
            location=[row['Centroid_Lat'], row['Centroid_Long']],
            radius=max(2000, row['Radius_km'] * 1000),
            color=color, fill=True, fill_opacity=0.22, weight=2,
            popup=f"Cluster {cid} — {row['Booking_Count']} bookings | {row['All_Pincodes']}",
        ).add_to(m)
        hull_coords = hull_polygons.get(cid)
        if hull_coords and len(hull_coords) >= 3:
            folium.Polygon(
                locations=hull_coords, color=color,
                fill=True, fill_opacity=0.35, weight=2,
                popup=f"Hull — Cluster {cid}",
            ).add_to(m)

    # All individual booking points for this city
    for _, row in city_data.iterrows():
        clr = '#dc2626' if row['Cluster'] >= 0 else '#94a3b8'
        folium.CircleMarker(
            [row['Lat'], row['Long']], radius=4,
            color=clr, fill=True, fill_opacity=0.7, weight=0,
            popup=f"{row['Area']} | Cluster {row['Cluster']} | {row['Pincode']}",
        ).add_to(m)

    fname = f"city_map_{city.lower().replace(' ', '_')}.html"
    m.save(fname)
    print(f"     {fname}  ({len(city_data)} bookings, {len(city_clust)} clusters)")

#12c.Plotly dashboard
city_counts   = df_clean['City'].value_counts().head(12)
severity_cts  = cluster_df['Severity'].value_counts().reindex(['Critical','High','Medium','Low']).fillna(0)
addr_counts   = df_clean['Address Type'].value_counts().head(6)
cluster_sizes = cluster_df['Booking_Count'].values

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Bookings by City (Top 12)', 'Cluster Severity Distribution', 'Cluster Size Distribution',
        'Address Type Breakdown', 'ε Cross-validation (Silhouette)', 'Financial Sensitivity'
    ),
    specs=[
        [{'type':'bar'}, {'type':'pie'}, {'type':'histogram'}],
        [{'type':'bar'}, {'type':'bar'}, {'type':'bar'}],
    ],
)

fig.add_trace(go.Bar(x=city_counts.index.tolist(), y=city_counts.values.tolist(),
                     marker_color='#7c3aed', name=''), row=1, col=1)

fig.add_trace(go.Pie(
    labels=severity_cts.index.tolist(),
    values=severity_cts.values.tolist(),
    marker=dict(colors=[SEVERITY_COLORS[s] for s in severity_cts.index]),
    textinfo='label+percent', hole=0.35,
), row=1, col=2)

fig.add_trace(go.Histogram(x=cluster_sizes.tolist(), nbinsx=12,
                           marker_color='#dc2626', name=''), row=1, col=3)

fig.add_trace(go.Bar(x=addr_counts.index.tolist(), y=addr_counts.values.tolist(),
                     marker_color='#059669', name=''), row=2, col=1)

# Epsilon CV bars
valid_cv_plot = cv_df.dropna(subset=['Silhouette'])
fig.add_trace(go.Bar(
    x=[f"ε={r}km" for r in valid_cv_plot['Epsilon_km']],
    y=valid_cv_plot['Silhouette'].tolist(),
    marker_color=['#f59e0b' if r == EPSILON_KM else '#6b7280' for r in valid_cv_plot['Epsilon_km']],
    name='',
), row=2, col=2)

# Sensitivity bars
fig.add_trace(go.Bar(
    x=sensitivity_df['Scenario'].tolist(),
    y=sensitivity_df['Cluster_Loss'].tolist(),
    marker_color=['#10b981', '#7c3aed', '#dc2626'],
    name='',
    text=[f"₹{v:,.0f}" for v in sensitivity_df['Cluster_Loss']],
    textposition='outside',
), row=2, col=3)

fig.update_layout(
    height=800, showlegend=False,
    title_text='COOX Analysis Dashboard v3',
    paper_bgcolor='#0f0f0f', plot_bgcolor='#1a1a1a',
    font=dict(color='#e5e5e5'),
)
fig.update_xaxes(gridcolor='#333')
fig.update_yaxes(gridcolor='#333')
fig.write_html('coox_analysis_dashboard.html')
print(" coox_analysis_dashboard.html  (6-panel, includes CV + sensitivity)")

#12d.All-bookings heatmap
all_bookings_map = folium.Map(location=[20.5, 78.9], zoom_start=5,
                              tiles='CartoDB positron')

heat_data_all = [[row['Lat'], row['Long'], 1] for _, row in df_clean.iterrows()]
HeatMap(
    heat_data_all,
    radius=18,
    blur=12,
    min_opacity=0.4,
    max_zoom=14,
    name='All Bookings Heatmap',
).add_to(all_bookings_map)

city_stats = df_clean.groupby('City').agg(
    Lat=('Lat', 'mean'), Long=('Long', 'mean'),
    Count=('Booking ID', 'count')
).reset_index()

for _, row in city_stats[city_stats['Count'] >= 10].iterrows():
    folium.Marker(
        [row['Lat'], row['Long']],
        popup=f"<b>{row['City']}</b><br>{row['Count']} bookings",
        icon=folium.Icon(color='blue', icon='info-sign'),
    ).add_to(all_bookings_map)

heatmap_legend = """
<div style="position:fixed;bottom:12px;left:12px;z-index:9999;
     background:white;padding:12px;border-radius:6px;
     border:1px solid #ccc;font-family:Arial;font-size:12px;
     box-shadow:2px 2px 6px rgba(0,0,0,.2)">
  <b>All Bookings Heatmap</b><br>
  Hotter = more bookings<br>
  Blue markers = cities with ≥10 bookings
</div>"""
all_bookings_map.get_root().html.add_child(folium.Element(heatmap_legend))
all_bookings_map.save('heatmap_all_bookings.html')
print("    heatmap_all_bookings.html")
print("coox_kdistance_plot.html")

# 13. FINAL CONSOLE SUMMARY
print(f"\n{'='*70}")
print("ANALYSIS COMPLETE — v3")
print(f"{'='*70}")
print(f"  Epsilon used           : {EPSILON_KM} km (auto-selected via cross-validation)")
print(f"  Clusters identified    : {n_clusters}")
print(f"  Bookings covered       : {n_clustered} / {len(df_clean)}  ({n_clustered/len(df_clean)*100:.1f}%)")
print(f"  Unique pincodes        : {len(all_block_pins)} to block")
print(f"  Watchlist pincodes     : {len(noise_pin_counts)}")
print(f"  Hull polygons          : {sum(1 for v in hull_polygons.values() if v)} / {n_clusters} clusters")
print(f"  Phase 1 pincodes       : {', '.join(phase1_pins[:10])}{'...' if len(phase1_pins)>10 else ''}")
print(f"\n  Financial (base case ₹{AVG_BOOKING_VALUE:,}):")
print(f"    Total addressable loss   : ₹{total_loss:,}")
print(f"    Recoverable via blocking : ₹{cluster_loss:,}  ({cluster_loss/total_loss*100:.1f}%)")
print(f"\n  Files generated:")
for f in [
    'coox_cluster_blocking_list.csv',
    'coox_bookings_with_clusters.csv',
    'coox_isolated_pincode_watchlist.csv',
    'coox_epsilon_crossvalidation.csv',
    'coox_financial_sensitivity.csv',
    'coox_analysis_summary.json  ← hull_polygon per cluster (engineering-ready)',
    'coox_cluster_map.html       ← circles + convex hull layer',
    'coox_analysis_dashboard.html ← 6-panel incl. CV + sensitivity',
    'coox_kdistance_plot.html    ← elbow method for ε justification',
    'coox_temporal_trend.html    ← month-over-month (if date col found)',
    'city_map_<city>.html        ← one per top city',
]:
    print(f"    • {f}")
print(f"\n  Next steps:")
print(f"    1. Review cluster_blocking_list.csv & validate top clusters on map")
print(f"    2. Add Phase 1 pincodes to booking-validation blacklist (dev: ~2 days)")
print(f"    3. Share hull_polygon JSON with engineering for precise geoblocking")
print(f"    4. Monitor for 1 week, then roll out Phase 2")
print(f"    5. Schedule monthly re-run of this script")
print(f"{'='*70}")


[11] SAVING OUTPUT FILES
    coox_cluster_blocking_list.csv   (63 clusters)
    coox_bookings_with_clusters.csv  (546 rows)
    coox_isolated_pincode_watchlist.csv (11 pins)
    coox_epsilon_crossvalidation.csv
    coox_financial_sensitivity.csv
    coox_analysis_summary.json  (includes hull_polygon per cluster)

[12] GENERATING VISUALISATIONS
     coox_cluster_map.html  (includes convex hull polygons)
     city_map_pune.html  (94 bookings, 11 clusters)
     city_map_ahmedabad.html  (38 bookings, 3 clusters)
     city_map_bengaluru.html  (143 bookings, 19 clusters)
     city_map_chennai.html  (66 bookings, 10 clusters)
     city_map_hyderabad.html  (68 bookings, 7 clusters)
     city_map_navi_mumbai.html  (14 bookings, 3 clusters)
     city_map_surat.html  (15 bookings, 1 clusters)
     city_map_kolkata.html  (18 bookings, 1 clusters)
     city_map_chandigarh.html  (9 bookings, 3 clusters)
     city_map_lucknow.html  (8 bookings, 1 clusters)
     city_map_jaipur.html  (25 bookings, 2 

In [16]:
# 12f. SEVERITY-SPLIT HEATMAP
from folium.plugins import HeatMap
import folium

severity_map = folium.Map(location=[20.5, 78.9], zoom_start=5,
                          tiles='CartoDB positron')

sev_colors_gradient = {
    'High':   {0.4: 'blue', 0.65: 'orange', 1: 'red'},
    'Medium': {0.4: 'blue', 0.65: 'yellow', 1: 'orange'},
    'Low':    {0.4: 'blue', 0.65: 'lime',   1: 'green'},
}

for sev in ['High', 'Medium', 'Low']:
    sev_cluster_ids = cluster_df[cluster_df['Severity'] == sev]['Cluster_ID'].tolist()
    sev_bookings = df_clean[df_clean['Cluster'].isin(sev_cluster_ids)]
    if sev_bookings.empty:
        continue
    heat_pts = [[row['Lat'], row['Long'], 1] for _, row in sev_bookings.iterrows()]
    HeatMap(
        heat_pts,
        name=f'{sev} severity clusters',
        radius=20, blur=15, min_opacity=0.45,
        gradient=sev_colors_gradient[sev],
    ).add_to(severity_map)

noise_heat = [[row['Lat'], row['Long'], 0.3] for _, row in noise_df.iterrows()]
HeatMap(
    noise_heat,
    name='Isolated bookings (noise)',
    radius=14, blur=10, min_opacity=0.2,
    gradient={0.4: 'grey', 1: 'blue'},
    show=False,
).add_to(severity_map)

city_stats_sev = df_clean.groupby('City').agg(
    Lat=('Lat', 'mean'), Long=('Long', 'mean'),
    Total=('Booking ID', 'count'),
    Clustered=('Cluster', lambda x: (x >= 0).sum())
).reset_index()

for _, row in city_stats_sev[city_stats_sev['Total'] >= 5].iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=(
            f"<b>{row['City']}</b><br>"
            f"Total outskirt bookings: {row['Total']}<br>"
            f"In clusters: {row['Clustered']}<br>"
            f"Isolated: {row['Total'] - row['Clustered']}"
        ),
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:10px;font-weight:bold;color:#1e293b;'
                f'background:white;padding:2px 5px;border-radius:3px;'
                f'border:1px solid #94a3b8;white-space:nowrap">'
                f'{row["City"]} ({row["Total"]})</div>'
            ),
            icon_size=(120, 20), icon_anchor=(60, 10),
        )
    ).add_to(severity_map)

legend_html = """
<div style="position:fixed;top:12px;right:12px;z-index:9999;
     background:white;padding:14px 18px;border-radius:8px;
     border:1px solid #ccc;font-family:Arial;font-size:12px;
     box-shadow:2px 2px 6px rgba(0,0,0,.25)">
  <b>COOX Outskirts Heatmap</b><br>
  <i style="font-size:11px">Non-serviceable bookings by severity</i><br><br>
  &#128308; <b>High</b> severity zones (&ge;10 bookings/cluster)<br>
  &#128992; <b>Medium</b> severity zones (5&ndash;9 bookings/cluster)<br>
  &#128994; <b>Low</b> severity zones (3&ndash;4 bookings/cluster)<br>
  &#9899; Isolated (noise) &mdash; toggle in layers<br><br>
  <i style="font-size:10px">Hotter color = higher booking density<br>
  Use layer control (top-left) to toggle severity</i>
</div>"""
severity_map.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(severity_map)
severity_map.save('coox_heatmap_severity.html')
print('\u2705 coox_heatmap_severity.html saved')

# 12g. PER-CITY STANDALONE HEATMAPS (top 6 cities)
city_totals = df_clean.groupby('City').agg(
    Lat=('Lat', 'mean'), Long=('Long', 'mean'),
    Total=('Booking ID', 'count'),
).reset_index()
top_cities = city_totals.sort_values('Total', ascending=False).head(6)['City'].tolist()

for city in top_cities:
    city_all = df_clean[df_clean['City'] == city]
    if len(city_all) < 3:
        continue
    center = [city_all['Lat'].mean(), city_all['Long'].mean()]
    cm = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')
    clustered_pts = city_all[city_all['Cluster'] >= 0]
    noise_pts = city_all[city_all['Cluster'] == -1]
    if not clustered_pts.empty:
        HeatMap(
            [[r['Lat'], r['Long'], 1] for _, r in clustered_pts.iterrows()],
            name='Clustered (blocked zones)',
            radius=25, blur=15, min_opacity=0.5,
            gradient={0.4: 'yellow', 0.65: 'orange', 1: 'red'},
        ).add_to(cm)
    if not noise_pts.empty:
        HeatMap(
            [[r['Lat'], r['Long'], 1] for _, r in noise_pts.iterrows()],
            name='Isolated bookings',
            radius=18, blur=12, min_opacity=0.3,
            gradient={0.4: 'blue', 1: 'cyan'},
            show=False,
        ).add_to(cm)
    folium.LayerControl(collapsed=False).add_to(cm)
    city_fname = f"coox_heatmap_{city.lower().replace(' ', '_')}.html"
    cm.save(city_fname)
    print(f'\u2705 {city_fname}')


✅ coox_heatmap_severity.html saved
✅ coox_heatmap_bengaluru.html
✅ coox_heatmap_pune.html
✅ coox_heatmap_hyderabad.html
✅ coox_heatmap_chennai.html
✅ coox_heatmap_ahmedabad.html
✅ coox_heatmap_jaipur.html
